In [13]:
import torch
from torch import nn
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer,AutoConfig,AutoModel
import numpy as np
import pandas as pd

In [14]:
model_ckpt = 'bert-base-uncased'
model = AutoModel.from_pretrained(model_ckpt)
print(model.encoder.layer[0])
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
config = AutoConfig.from_pretrained(model_ckpt)
config

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.12.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [15]:
def scaled_dot_attention(query,key,value):
  dim_k = key.size(-1)
  attn_scores = torch.bmm(query,key.transpose(1,2) / np.sqrt(dim_k))
  attn_weights = F.softmax(attn_scores,dim=-1)
  return torch.bmm(attn_weights,value)

In [16]:
class AttentionHead(nn.Module):
  def __init__(self,embed_dim,head_dim):
    super().__init__()
    self.Wq = nn.Linear(embed_dim,head_dim)
    self.Wk = nn.Linear(embed_dim,head_dim)
    self.Wv = nn.Linear(embed_dim,head_dim)
  def forward(self,hidden_states):
    q = self.Wq(hidden_states)
    k = self.Wk(hidden_states)
    v = self.Wv(hidden_states)
    attn_outputs = scaled_dot_attention(q,k,v)
    return attn_outputs

In [17]:
class MultiHeadAttention(nn.Module):
  def __init__(self,config):
    super().__init__()
    embed_dim = config.hidden_size
    num_heads = config.num_attention_heads

    head_dim = embed_dim // num_heads
    self.heads = nn.ModuleList({
        AttentionHead(embed_dim,head_dim) for _ in range(num_heads)
    })
    self.output_layer = nn.Linear(embed_dim,embed_dim)
  def forward(self,hidden_state):
    print(f'input hidden_state:{hidden_state.shape}')
    print(f'head(hidden_state):{self.heads[11](hidden_state).shape}')
    x = torch.cat([head(hidden_state) for head in self.heads],dim=-1)
    print(f'x.shape:{x.shape}')
    x = self.output_layer(x)
    return x

In [18]:
mha = MultiHeadAttention(config)

In [19]:
token_embedding = nn.Embedding(config.vocab_size,config.hidden_size)
sample_text = 'time flies like an arrow'
model_inputs = tokenizer(sample_text,return_tensors='pt',add_special_token=False)
input_embeddings = token_embedding(model_inputs['input_ids'])
input_embeddings.shape

torch.Size([1, 7, 768])

In [20]:
attn_output = mha(input_embeddings)

input hidden_state:torch.Size([1, 7, 768])
head(hidden_state):torch.Size([1, 7, 64])
x.shape:torch.Size([1, 7, 768])


# FFN

In [12]:
class FeedForward(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.fc1 = nn.Linear(config.hidden_size,config.intermediate_size)
    self.fc2 = nn.Linear(config.intermediate_size,config.hidden_size)
    self.gelu = nn.GELU()
    self.dropout = nn.Dropout(config.hidden_dropout_prob)
  def forward(self,x):
    # b s h => b s 4h
    x = self.fc1(x)
    x = self.gelu(x)
    # b s 4h => b s h
    x = self.fc2(x)
    x = self.dropout(x)
    return x

In [21]:
ffn = FeedForward(config)

In [22]:
ffn(attn_output).size()

torch.Size([1, 7, 768])

# EncoderLayer

In [23]:
class TransformerEncoderLayer(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.ln1 = nn.LayerNorm(config.hidden_size)
    self.ln2 = nn.LayerNorm(config.hidden_size)
    self.attn = MultiHeadAttention(config)
    self.ffn = FeedForward(config)
  def forward(self,x):
    x = x + self.attn(self.ln1(x))
    x = x + self.ffn(self.ln2(x))
    return x


In [24]:
encoder_layer = TransformerEncoderLayer(config)


In [25]:
encoder_layer(input_embeddings).shape

input hidden_state:torch.Size([1, 7, 768])
head(hidden_state):torch.Size([1, 7, 64])
x.shape:torch.Size([1, 7, 768])


torch.Size([1, 7, 768])